# 01. Multi-Modal Image Understanding with Azure AI Foundry

This notebook demonstrates **multimodal input** — sending an image *and* text together in a single
request — to a vision-capable model deployed behind an Azure AI Foundry project. It uses the
Foundry project's OpenAI-compatible `responses` API to ask a model to summarize the contents of a
chart image (`sales_data.png`). This belongs to the *Image Generation & Content Safety* section of
the course, specifically the "understanding images" half (as opposed to *generating* or *editing*
images, covered in notebooks 02-04).

**Difficulty:** Beginner / Intermediate — assumes basic familiarity with Azure AI Foundry projects
(see `11_azure_ai_foundry/00_setup/`) but nothing about multimodal APIs specifically.

## Prerequisites

**Python packages** (already in the repo's root `requirements.txt` — no extra install needed):
- `azure-ai-projects` — `AIProjectClient`
- `azure-identity` — `DefaultAzureCredential`
- `python-dotenv` — loads `.env`
- `base64` — Python standard library, no install required

**Azure resources**
- An Azure AI Foundry **project** with a **vision-capable / multimodal model deployment**
  (e.g. a GPT-4o-class or newer model that accepts image input). The original script used a
  deployment named `gpt-5.4` — substitute whatever multimodal deployment name exists in your
  own Foundry project.

**Environment variables** — add these to the root `.env`:
```
AZURE_AI_PROJECT_ENDPOINT=https://<your-foundry-resource>.services.ai.azure.com/api/projects/<your-project>
AZURE_AI_MODEL_DEPLOYMENT=<your-multimodal-deployment-name>
```
These follow the same naming convention already used in `11_azure_ai_foundry/`. This notebook
reuses those two variables rather than inventing new ones, since it talks to the same kind of
Foundry project resource.

> This repo's `.env` already has `AZURE_AI_PROJECT_ENDPOINT` and `AZURE_AI_MODEL_DEPLOYMENT` set —
> just confirm the deployment behind that name actually accepts image input before running this
> notebook for real.

## What You'll Learn

- How to obtain a standard OpenAI-SDK-compatible client from an Azure AI Foundry project via
  `project.get_openai_client()`
- How to encode a local image file as base64 and embed it as a `data:` URI
- The `responses.create()` content-part format for multimodal input (`input_text` +
  `input_image`), as distinct from the older `chat.completions` message format
- Why `DefaultAzureCredential` (Entra ID / `az login`) is used instead of an API key for
  Foundry project access
- How the model's answer is read back via `response.output_text`

### Imports and setup

- `base64` — used to convert the raw image bytes into a base64 text string, since JSON (and the
  `data:` URI scheme) can't carry raw binary directly.
- `AIProjectClient` — the top-level handle to your Azure AI Foundry **project** (not a single
  Azure OpenAI resource — a project can bundle multiple model deployments, agents, and
  connections).
- `DefaultAzureCredential` — authenticates using your `az login` session (Entra ID) rather than
  an API key, which is the standard auth pattern for Foundry projects in this repo.
- `load_dotenv()` reads the root `.env` file into `os.environ` so `os.getenv(...)` calls below
  can find the values.

💡 **Exam tip:** AI-102 expects you to know that Azure AI Foundry (and Azure AI services broadly)
supports **both** API-key auth and Microsoft Entra ID (Azure AD) token-based auth via
`DefaultAzureCredential` / `azure-identity`. Entra ID is the recommended approach for
production workloads because it avoids storing long-lived secrets.

🔄 **Alternatives:** You could instead construct a plain `AzureOpenAI` client directly against a
single Azure OpenAI resource (bypassing the Foundry *project* abstraction entirely) if you don't
need project-level features like connected agents or multiple deployments.

In [ ]:
import os
import base64

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

### Building the Foundry project client

- `PROJECT_ENDPOINT` is read from the environment instead of being hardcoded.
- `MODEL_DEPLOYMENT` falls back to `"gpt-5.4"` (the original script's hardcoded value) if the env
  var isn't set — but for this to actually run, it must name a deployment that exists in your
  project **and** accepts image input.
- `AIProjectClient(endpoint, credential)` — creates the project-level client.
- `project.get_openai_client()` — hands back a client with the same interface as the standard
  `openai` Python SDK (`.responses.create(...)`, `.chat.completions.create(...)`, etc.), but
  pre-wired to talk to your Foundry project's deployments.

💡 **Exam tip:** `get_openai_client()` is the bridge between the Azure-specific
`azure-ai-projects` SDK and the OpenAI-compatible surface — this is how Foundry lets you reuse
OpenAI SDK code/knowledge instead of learning an entirely separate calling convention.

🔄 **Alternatives:** For simple stateless calls like this one, you could also use the Foundry
project's REST API directly (no SDK) — useful in languages without an `azure-ai-projects` SDK.

In [ ]:
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT", "gpt-5.4")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

openai = project.get_openai_client()

### Reading and base64-encoding the image

- Opens `sales_data.png` (already present alongside this notebook) in binary mode (`"rb"`).
- `base64.b64encode(...)` converts the raw bytes into a base64 byte string; `.decode("utf-8")`
  turns that into a normal Python string so it can be embedded inside a JSON request body and,
  further down, inside a `data:` URI.

💡 **Exam tip:** Sending images inline as base64 `data:` URIs avoids needing a publicly
reachable image URL — useful for local files or private/internal images. Many Azure AI vision
APIs accept *either* a hosted URL *or* inline base64 bytes.

🔄 **Alternatives:** Instead of inlining the image, you could first upload it somewhere
accessible (e.g. Azure Blob Storage with a SAS URL) and pass that URL as `image_url` instead —
more efficient for large images or images reused across multiple requests.

In [ ]:
IMAGE_PATH = "sales_data.png"

with open(IMAGE_PATH, "rb") as f:
    b64 = base64.b64encode(f.read()).decode("utf-8")

### Sending the multimodal request

- `openai.responses.create(...)` calls the Responses API (OpenAI's newer unified API surface,
  which Azure OpenAI/Foundry also implements).
- `input` is a list of message dicts. Each message's `content` is itself a list of **content
  parts** — here, one `input_text` part (the question) and one `input_image` part (the image,
  passed as a `data:image/png;base64,<...>` URI).
- This differs from the older `chat.completions` format, where multimodal content is nested
  under `messages[i]["content"]` using `{"type": "text", ...}` / `{"type": "image_url", ...}`
  instead of `input_text` / `input_image`.

💡 **Exam tip:** AI-102 covers multimodal (vision-enabled) chat models as part of Azure OpenAI /
Azure AI Foundry's generative AI capabilities — know that vision input can be supplied either as
a hosted URL or as base64-encoded inline data, and that not every model deployment supports
image input (it depends on the specific model).

🔄 **Alternatives:** For a single static image + question like this, you could alternatively use
Azure AI Vision's **Image Analysis** API (captioning, OCR, tags) if you want structured,
non-generative output rather than a free-form LLM-generated summary.

In [ ]:
response = openai.responses.create(
    model=MODEL_DEPLOYMENT,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text",
             "text": "Generate a summary based on the content in the attached image"},
            {"type": "input_image",
             "image_url": f"data:image/png;base64,{b64}"},
        ],
    }],
)

### Reading the result

`response.output_text` is a convenience property on the Responses API result that concatenates
the model's generated text output into a single string, so you don't have to manually walk the
`response.output` list of content blocks.

In [ ]:
print(response.output_text)

## Summary

This notebook sent a chart image (`sales_data.png`) plus a text instruction to a multimodal model
deployment via an Azure AI Foundry project's OpenAI-compatible `responses.create()` call. The
model returned a natural-language summary of what it "saw" in the image, printed via
`response.output_text`. The key mechanics were: authenticating with `DefaultAzureCredential`,
obtaining an OpenAI-compatible client from the Foundry project, and encoding the local image as a
base64 `data:` URI inside an `input_image` content part.

## Try It Yourself

1. **Easy:** Swap `sales_data.png` for `support.png` (also in this folder) and change the prompt
   to ask a different question about the image (e.g. "What error is shown in this screenshot?").
2. **Intermediate:** Ask a follow-up question in a second `responses.create()` call, passing the
   first response back as conversation history (multi-turn multimodal chat).
3. **Advanced:** Compare this model-generated summary against Azure AI Vision's Image Analysis
   captioning API output for the same image — how do the two differ in accuracy vs. flexibility?